# §58 — ablation on the §55 DoTA-train corpus: **K5dfiDoTA** vs **K0dfiDoTA** (KIM off) vs **L0dfiDoTA** (LaGoVAD trunk), s2024

Rule: `core/docs/final/KIM_RUN.md` **§58**, written before any number. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. **Needs §55 done** (`k5dfidota.ipynb`). PRELIMINARY (n = 1). **Decides nothing; K5f stays the reference. DoTA-CAP-dev is in-domain for every row.**

| row | = | input | here |
|---|---|---|---|
| **K5dfiDoTA** | §55 (VideoMAE V2-S + KIM, DVS 0.7 `ignore`) | DoTA-statistics bake | reused (re-trained as `K5dfiDoTArr` only if the start gate fails) |
| **K0dfiDoTA** | K5dfiDoTA − the four `kim.*` flags | same bake | train + read |
| **L0dfiDoTA** | K0dfiDoTA − `v2.motion`, `model.motion_dim` (LaGoVAD trunk) | plain CLIP `DoTA_train_s3_ncc` | train + read |

**KIM = K5 − K0 · VideoMAE = K0 − L0 · total = K5 − L0**, on DoTA-CAP-dev (in-domain) and DADA (zero-shot). Labels §53 KA5: GAIN (≥ +0.01, interval above 0) / COST / NO DIFFERENCE; micro first.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-4 | setup, deps, helpers, preflight | no | 3-5 min |
| 5 | tests — **raises if red** | no | 1-2 min |
| 6 | stage corpus, bake, plain CLIP, KNN; **same CLIP frames** check (train, DoTA-CAP-dev, DADA) | no | 10-15 min |
| 7 | train K0dfiDoTA -> start gate (-> K5dfiDoTArr if needed) -> L0dfiDoTA (1,764 steps each) | **yes** | 2-3 × K5dfiDoTA's time |
| 8 | reads: DoTA-CAP-dev + DADA per new row | yes | 15-25 min per row |
| 9 | two tables + labels -> **paste `REPORTS/k5dfidota_ablation_report.md` to Claude** | no | 3-5 min |

A re-run skips what is on Drive (a row is skipped once `verified.json` matches). **Watch the log: past 2 s/step, stop and paste it.** Thesis / Thesis-V2 are only read.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip (DoTA_s1_ncc, DADA2000_orig)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window defs
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, §55 artifacts, K5dfiDoTA; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 58. Ablation on the §55 DoTA-train corpus' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §58 -- re-upload kat-vad/'
assert 'if not is_abnormal:\n            return NORMAL_CLASS' in (REPO / 'core/data/dataset.py').read_text(), \
    'core/data/dataset.py has no §31 fix -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'KIM_DOTA_TRAIN_OFF_ROW'), 'constants has no §58 rows -- re-upload kat-vad/'

K5       = constants.KIM_MAIN_ARM                  # "K5": names run / eval sub-folders only
CTRL     = constants.KIM_DOTA_TRAIN_ROW            # K5dfiDoTA (§55, reused)
RERUN    = CTRL + constants.KIM_ABLATION_RERUN_SUFFIX   # K5dfiDoTArr: only if the start gate fails (§58 AB3)
K0       = constants.KIM_DOTA_TRAIN_OFF_ROW        # K0dfiDoTA
L0       = constants.KIM_DOTA_TRAIN_L0_ROW         # L0dfiDoTA
KIM_FLAGS = constants.KIM_ABLATION_FLAGS
IGN      = constants.DVS_ANCHOR_MODE_IGNORE
SEED     = constants.KIM_SEED
ENC      = constants.V5_ENCODER                    # V2-S
OFF      = constants.V2_OFF
DATASET  = constants.NEXAR_DATASET
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
STRIDE_B = constants.V2_E1_STRIDE_BC               # DoTA s1[::3] = protocol B rows
NEX, DST_SUF = constants.NEXAR_STATS_BAKE_SUFFIX, constants.DOTA_TRAIN_STATS_BAKE_SUFFIX

WINDOW   = Path(DRIVE_V2) / 'data' / f'{DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'   # defs of every read
ROOT     = constants.OUTPUT_ROOT / constants.KIM_DOTA_TRAIN_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
BUILD    = REPORTS / constants.DOTA_TRAIN_BUILD_REPORT
ZS_SRC_NAMES = {'dada': f'{T2NAME}_test_sources_nexar', 'dota': f'{CAP_DEV}_s3_from_s1_nexar'}   # = §55's ZS_SRC
assert all(n.endswith(NEX) for n in ZS_SRC_NAMES.values())
ZS_DST   = {s: constants.CACHE_ROOT / 'v2' / 'A2_S' / (n[: -len(NEX)] + DST_SUF) for s, n in ZS_SRC_NAMES.items()}
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
T2_CLIP  = Path(DRIVE) / 'cache' / 'clip' / T2NAME                 # plain CLIP, L0dfi's DADA input
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
DATA     = Path(DRIVE_F) / 'data' / constants.DOTA_TRAIN_CORPUS_DIRNAME
BAKE     = constants.CACHE_ROOT / 'v2' / 'A2_S' / constants.DOTA_TRAIN_BAKE_DIRNAME
CLIP_S3  = constants.CLIP_CACHE_DIR / constants.DOTA_TRAIN_CLIP_DATASET   # plain CLIP s3 of DoTA train (§55)
KNN      = constants.CACHE_ROOT / 'knn' / DATA.name / constants.KNN_CACHE_FILENAME
LOCAL    = Path('/content/k5dfidota_abl')
LOCAL.mkdir(parents=True, exist_ok=True)


def row_paths(row):
    run_dir = ROOT / row / K5 / f's{SEED}'
    return run_dir, run_dir / 'checkpoint_last.pt', ROOT / row / 'eval'


CTRL_RUN, _, CTRL_EV = row_paths(CTRL)
INPUTS = (('§55 build report', BUILD), ('§55 corpus windows', DATA / constants.WINDOWS_FILENAME),
          ('§55 DoTA-train bake', BAKE / constants.V2_INPUT_MANIFEST_FILENAME), ('§55 KNN', KNN), ('§55 plain CLIP s3', CLIP_S3),
          *[(f'§55 restated {s} bake', d / constants.V2_INPUT_MANIFEST_FILENAME) for s, d in ZS_DST.items()],
          ('K5dfiDoTA checkpoint', CTRL_RUN / 'checkpoint_last.pt'), ('K5dfiDoTA verified', CTRL_RUN / 'verified.json'),
          ('K5dfiDoTA config', CTRL_RUN / 'config.yaml'), ('K5dfiDoTA metrics', CTRL_RUN / 'metrics.jsonl'),
          ('K5dfiDoTA DoTA read', CTRL_EV / 'dota' / K5 / 'clip_scores.npz'), ('K5dfiDoTA DADA read', CTRL_EV / 'dada' / K5 / 'clip_scores.npz'),
          ('Nexar_window defs', WINDOW / constants.DEFS_FILENAME), ('T2 meta', T2 / constants.META_FILENAME),
          ('T2 census', T2_CENSUS), ('T2 plain CLIP', T2_CLIP), ('DADA annotation', ANNOT), ('DoTA s1', DOTA_S1),
          ('DoTA metadata', DOTA / 'metadata_val.json'))
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing (run k5dfidota.ipynb first?) -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | new rows {K0}, {L0} (+ {RERUN} if the start gate fails) -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 7, 8 need one'
df -h /content | tail -1

In [ ]:
import numpy as np  # noqa: E402
import torch  # noqa: E402
import yaml  # noqa: E402

from core.config import Config                       # noqa: E402
from core.data.v2_splits import load_split           # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402
from core.tools.dota_train_extract import train_clips  # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2)."""
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    names = [f.name for f in src.iterdir() if f.is_file()] if ids is None else \
        [f'{v}.npy' for v in ids] + [m.name for p in ('*.json', '*.npz') for m in src.glob(p)]
    todo = [n for n in names if not (dst / n).exists()]
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda n: shutil.copy2(src / n, dst / n), todo))
    print(f'  staged {src.name}: {len(todo)} files in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def on_drive(folder, min_bytes=1024):
    """Ids of every finished ``.npy`` in ``folder`` from ONE listing (save_array = tmp + rename)."""
    if not folder.is_dir():
        return set()
    with os.scandir(folder) as entries:
        return {e.name[:-4] for e in entries if e.name.endswith('.npy') and e.stat().st_size >= min_bytes}


def metrics_rows(path):
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()] if path.exists() else []


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(run_dir, steps):
    rec, ckpt = run_dir / 'verified.json', run_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())


DEFAULTS = _flat(Config().to_dict())


def config_diff(base, got):
    """Keys that differ; a key added after ``base`` trained may sit at its default."""
    out = {}
    for k in set(base) | set(got):
        if base.get(k) == got.get(k):
            continue
        if k not in base and got.get(k) == DEFAULTS.get(k):
            continue
        if k not in got and base.get(k) == DEFAULTS.get(k):
            continue
        out[k] = (base.get(k), got.get(k))
    return out

from core.data.v2_splits import lines_sha1          # noqa: E402
from core.tools import k5_edge_report as edge          # noqa: E402
from core.tools import kim_ablation_report as ka       # noqa: E402

# Preflight on small files only (C39)
ctrl_cfg = _flat(yaml.safe_load((CTRL_RUN / 'config.yaml').read_text()))
assert all(ctrl_cfg[k] for k in KIM_FLAGS), 'CTRL_RUN is not K5dfiDoTA'
assert (ctrl_cfg['dvs.theta'], ctrl_cfg['loss.dvs_anchor_mode'], ctrl_cfg['data.is_egocentric'], ctrl_cfg['data.dataset'],
        ctrl_cfg['loss.bottomk_weight'], ctrl_cfg['v2.motion']) == (0.7, IGN, False, constants.DOTA_TRAIN_DATASET, 0.0, ENC), \
    'CTRL_RUN is not K5dfiDoTA'
build = json.loads(BUILD.read_text())
info, PLAN = build['construction'], build['plan']
assert (info['train_items'], info['train_abnormal'], info['train_normal'], info['clips_with_windows']) == (12937, 9386, 3551, 3019), f'not §55 DC2: {info}'
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']
assert (STEPS, EPOCHS) == (1764, 6), PLAN
assert is_verified(CTRL_RUN, STEPS) and metrics_rows(CTRL_RUN / 'metrics.jsonl')[-1]['global_step'] == STEPS, 'K5dfiDoTA: J10'
TRAIN = train_clips(constants.DOTA_TRAIN_SPLITS_DIR / constants.DOTA_TRAIN_MAP_FILENAME)
ids_all = sorted(TRAIN)
windows = json.loads((DATA / constants.WINDOWS_FILENAME).read_text())
TRAIN_SRC = sorted({w['source'] for w in windows.values()})
SHA_DOTA = lines_sha1(TRAIN_SRC)
m = manifest(BAKE)
assert (m['crn'], m['motion'], m['train_ids_sha1'], m['baked_ids']) == (OFF, ENC, SHA_DOTA, len(ids_all)), f'bake: {m}'
DADA_IDS, CAP_IDS = test_sources(T2), sorted(load_split(CAP_DEV))
ZS_IDS = {'dada': DADA_IDS, 'dota': CAP_IDS}
for s, d in ZS_DST.items():
    mm = manifest(d)
    assert (mm['crn'], mm['motion'], mm['train_ids_sha1'], mm['baked_ids']) == (OFF, ENC, SHA_DOTA, len(ZS_IDS[s])), f'{s}: {mm}'
assert len(TRAIN) == 3020 and len(TRAIN_SRC) == 3019 and not set(TRAIN) & set(CAP_IDS)
print(f'preflight OK: K5dfiDoTA step {STEPS} | corpus {info} | DoTA-CAP-dev {len(CAP_IDS)}, DADA {len(DADA_IDS)}')

## 5. Tests on the touched code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_dota_train_corpus.py',
     'core/tests/test_kim.py', 'core/tests/test_dvs_dataset.py', 'core/tests/test_zs_compare.py',
     'core/tests/test_kim_ablation_report.py'])

## 6. Stage the §55 corpus, bake, plain CLIP and KNN; same CLIP frames for every row (§58 AB4)

In [ ]:
L_DATA  = stage(DATA, LOCAL / 'stage' / 'data')
L_BAKE  = stage(BAKE, LOCAL / 'stage' / 'bake', ids_all)
L_X     = stage(CLIP_S3, LOCAL / 'stage' / 'plain_s3', ids_all)          # L0dfiDoTA's training input
L_KNN   = stage(KNN, LOCAL / 'stage' / 'knn' / KNN.name)
L_DOTA_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS)
L_T2CLIP  = stage(T2_CLIP, LOCAL / 'stage' / 'dada_clip', DADA_IDS)
L_ZS = {s: stage(d, LOCAL / 'stage' / f'zs_{s}', ZS_IDS[s]) for s, d in ZS_DST.items()}
assert (DATA / constants.DEFS_FILENAME).read_text() == (WINDOW / constants.DEFS_FILENAME).read_text(), 'defs differ from Nexar_window'
assert len(list(L_BAKE.glob('*.npy'))) == len(ids_all) == len(list(L_X.glob('*.npy'))), 'staged partially -- re-run this cell'

# §58 AB4: every row sees the same CLIP frames (plain rows == the bakes' first 512 columns)
CHECKS = (('train', L_X, L_BAKE, ids_all, 1), ('dota', L_DOTA_S1, L_ZS['dota'], CAP_IDS, STRIDE_B),
          ('dada', L_T2CLIP, L_ZS['dada'], DADA_IDS, 1))
for name, plain, bake, ids, stride in CHECKS:
    done = LOCAL / 'checks' / f'{name}.ok'
    if done.exists():
        print(f'{name}: checked earlier this session')
        continue
    run([sys.executable, '-m', 'core.tools.zs_lagovad_readout', 'check-inputs', '--plain', plain, '--bake', bake,
         '--ids-file', write_ids(LOCAL / 'ids' / f'{name}.txt', ids), '--plain-stride', stride])
    done.parent.mkdir(parents=True, exist_ok=True)
    done.write_text('ok')
print(f'staged + same CLIP frames: train {len(ids_all)}, DoTA-CAP-dev {len(CAP_IDS)}, DADA {len(DADA_IDS)} | KNN {L_KNN.stat().st_size / 2**20:.0f} MB')

## 7. Train **K0dfiDoTA**, start gate, then **L0dfiDoTA** (§58 AB1 / AB3 / AB4)

K0dfiDoTA = K5dfiDoTA's command with the four `kim.*` flags false. Its step-1 `mil` must equal K5dfiDoTA's (KA3); if not, K5dfiDoTA's exact command is re-trained here as **K5dfiDoTArr** and that is the KIM-on side. L0dfiDoTA = K0dfiDoTA + `v2.motion=none`, `model.motion_dim=0` on the plain CLIP s3 cache.

In [ ]:
BASE_SETS = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={EPOCHS}', 'train.amp=true',
             f'data.dataset={constants.DOTA_TRAIN_DATASET}', f'data.frame_stride={constants.DOTA_TRAIN_STRIDE}',
             f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}', 'kip.enabled=false', f'loss.dvs_anchor_mode={IGN}',
             'data.is_egocentric=false', f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
             'dvs.theta=0.7', 'dvs.theta_ego=0.85']   # = K5dfiDoTA's command (§55 notebook, step 8) without motion / KIM
MOTION = [f'v2.motion={ENC}', f'model.motion_dim={constants.V5_GRID_DIM}']
KIM_ON = [f'{k}=true' for k in KIM_FLAGS]
KIM_OFF = [f'{k}=false' for k in KIM_FLAGS]
SPEC = {RERUN: ([*BASE_SETS, *MOTION, *KIM_ON], 'bake'),
        K0: ([*BASE_SETS, *MOTION, *KIM_OFF], 'bake'),
        L0: ([*BASE_SETS, f'v2.motion={OFF}', 'model.motion_dim=0', *KIM_OFF], 'plain')}
INPUT = {'bake': L_BAKE, 'plain': L_X}


def last_epoch(rows, key):
    last = [r for r in rows if r['epoch'] == rows[-1]['epoch']]
    return sum(r.get(key, float('nan')) for r in last) / len(last)


def check_run(row):
    run_dir, ckpt, _ = row_paths(row)
    mrows = metrics_rows(run_dir / 'metrics.jsonl')
    assert ckpt_step(ckpt) == mrows[-1]['global_step'] == STEPS, f'{row}: J10'
    got = _flat(yaml.safe_load((run_dir / 'config.yaml').read_text()))
    if row == RERUN:
        diff = ka.config_diff(ctrl_cfg, got)
        assert not diff, f'{row}: differs from {CTRL} by {diff} (KA3 allows nothing)'
    elif row == K0:
        print(f'{row}: KIM flags off vs {CTRL}:', ka.check_kim_only(ctrl_cfg, got, row))
    else:
        k0_cfg = _flat(yaml.safe_load((row_paths(K0)[0] / 'config.yaml').read_text()))
        diff = ka.config_diff(k0_cfg, got)
        assert set(diff) == {'v2.motion', 'model.motion_dim'}, f'{row} vs {K0}: {diff} (§58 AB4)'
        print(f'{row}: diff to {K0} {diff}')
    mul = last_epoch(mrows, 'mul_mil')
    assert mul > constants.KIM_FIXED_MUL_MIL_FLOOR, f'{row}: last-epoch mul_mil {mul:.4f} -- paste to Claude'
    ref = metrics_rows(CTRL_RUN / 'metrics.jsonl')
    for key in ('mil', 'mul_mil', 'kim_mil'):
        print(f'  last-epoch {key} {last_epoch(mrows, key):.4f} ({CTRL} {last_epoch(ref, key):.4f})')


def train_row(row):
    run_dir, ckpt, _ = row_paths(row)
    if is_verified(run_dir, STEPS):
        print(f'{row} s{SEED}: verified on Drive -- not retrained')
        check_run(row)
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    sets, inp = SPEC[row]
    local = LOCAL / 'runs' / row
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    env = {'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': str(torch.version.cuda)}
    print(f'{row}: trains on {env}')
    (local / constants.KIM_RUN_ENV_FILENAME).write_text(json.dumps(env))
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in sets for x in ('--set', s)], '--data-dir', L_DATA,
         '--clip-dir', INPUT[inp], '--output-dir', local, '--knn-cache', L_KNN, *resume], log=LOCAL / f'train_{row}.log')
    last = metrics_rows(local / 'metrics.jsonl')[-1]
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{row}: J10 locally {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml', constants.KIM_RUN_ENV_FILENAME):
        publish(local / name, run_dir / name)
    flush_drive()
    assert ckpt_step(ckpt) == STEPS, f'{row}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': ckpt.stat().st_size}))
    publish(local / 'verified.json', run_dir / 'verified.json')
    print(f'{row}: trained in {(time.time() - t0) / 60:.0f} min')
    check_run(row)


# K0dfiDoTA, then the start gate (KA3)
train_row(K0)
st = edge.start_check(CTRL_RUN, row_paths(K0)[0])
print(f"KA3 step 1: {K0} {st['new']:.6f} vs saved {CTRL} {st['control']:.6f} -> {'same start' if st['same_draws'] else 'DIFFERENT START'}")
ON, START = CTRL, 'matched (saved K5dfiDoTA)'
if not st['same_draws'] or is_verified(row_paths(RERUN)[0], STEPS):
    print(f'{CTRL}: its start is not matched here -> re-train its exact command ({RERUN})')
    train_row(RERUN)
    st = edge.start_check(row_paths(RERUN)[0], row_paths(K0)[0])
    ON = RERUN
    START = 'matched (re-run)' if st['same_draws'] else 'UNMATCHED START'
    print(f"KA3 {K0} {st['new']:.6f} vs {RERUN} {st['control']:.6f} -> {START}")
KA3 = {'on_row': ON, 'status': START, 'step1_off': st['new'], 'step1_on': st['control']}

# L0dfiDoTA (no matched start: another model, printed only, §36 R4)
train_row(L0)
s0 = edge.start_check(row_paths(ON)[0], row_paths(L0)[0])
print(f"step 1 (printed): {L0} {s0['new']:.6f} vs {ON} {s0['control']:.6f}")
print('KIM-on side:', KA3)

## 8. Reads per new row: DoTA-CAP-dev (protocol B, **in-domain**) and DADA whole — K rows on the DoTA-statistics bakes, L0 on plain CLIP

In [ ]:
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in WINDOW.glob('*.json'):             # the definitions every row read with
    shutil.copy2(f, L_DEFS / f.name)
L_LAB = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')


def read_dota(row, plain):
    _, ckpt, ev = row_paths(row)
    out = ev / 'dota' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dota'
        shutil.rmtree(local, ignore_errors=True)
        inp, stride = (L_DOTA_S1, ['--stride', STRIDE_B]) if plain else (L_ZS['dota'], [])
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', ckpt, '--input-dir', inp,
             '--s1-dir', L_DOTA_S1, '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
             '--data-dir', L_LAB, '--split', CAP_DEV, *stride, '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{row}_dota.log')
        publish(local, out)
    print(f"{row} DoTA-CAP-dev: macro {json.loads((out / 'protocol_b_readout.json').read_text())['macro']['mean']:.4f}")


def read_dada(row, plain):
    _, ckpt, ev = row_paths(row)
    out = ev / 'dada' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / row / 'dada'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', ckpt, '--t2-dir', L_T2,
             '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', L_T2CLIP if plain else L_ZS['dada'],
             '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'{row}_dada.log')
        publish(local, out)
    print(f"{row} DADA whole: macro {json.loads((out / 'dada_readout.json').read_text())['macro']['mean']:.4f}")


ROWS = [ON, K0, L0]                         # ON = K5dfiDoTA (reads reused) or K5dfiDoTArr
for row in ROWS:
    read_dota(row, plain=row == L0)
    read_dada(row, plain=row == L0)
flush_drive()

## 9. Two tables (control K0dfiDoTA; control L0dfiDoTA) + KA5 labels -> **paste `REPORTS/k5dfidota_ablation_report.md` to Claude**

In [ ]:
local = LOCAL / 'report'
shutil.rmtree(local, ignore_errors=True)
local.mkdir(parents=True)
SETS = ('dota', 'dada')
SET_NAME = {'dota': 'DoTA-CAP-dev (in-domain)', 'dada': 'DADA-2000 (zero-shot)'}
TABLES = constants.KIM_DOTA_TRAIN_ABL_TABLES
for key, ctrl, rows in (('kim', K0, (ON, L0)), ('motion', L0, (K0, ON))):
    reads = [a for row in (ctrl, *rows) for s in SETS for a in ('--read', s, row, row_paths(row)[2] / s / K5)]
    run([sys.executable, '-m', 'core.tools.zs_arm_table', '--control', ctrl, '--seed', SEED, *reads,
         '--name', TABLES[key], '--out-dir', local])
tab = {k: json.loads((local / f'{constants.ZS_ARM_TABLE_STEM}_{n}.json').read_text())['sets'] for k, n in TABLES.items()}


def label(c):
    # §53 KA5
    if c['mean'] >= constants.V5_MIN_GAIN and c['low'] > 0:
        return constants.KIM_DVS_IGNORE_GAIN
    if c['mean'] <= -constants.V5_MIN_GAIN and c['high'] < 0:
        return constants.KIM_DVS_IGNORE_COST
    return constants.KIM_DVS_IGNORE_NO_DIFFERENCE


def cell(c, unmatched=False):
    return f"{c['mean']:+.4f} [{c['low']:+.4f}, {c['high']:+.4f}] {'UNMATCHED START' if unmatched else label(c)}"


CONTRASTS = (('KIM', f'{ON} − {K0}', 'kim', ON, KA3['status'] == 'UNMATCHED START'),
             ('VideoMAE motion', f'{K0} − {L0}', 'motion', K0, False),
             ('total (motion + KIM)', f'{ON} − {L0}', 'motion', ON, False))
pr, pl, ph = constants.KIM_DOTA_TRAIN_POSITION_RULER
dr, dl, dh = constants.KIM_DADA_MIDDLE_RULER
lines = ['# §58 ablation on DoTA train (KIM_RUN.md §58), s2024, n = 1 — printed, decides nothing; K5f stays', '',
         f"Corpus {info}; {STEPS} steps per row. KIM-on side **{ON}** — start gate (KA3): **{KA3['status']}** "
         f"(step-1 mil off {KA3['step1_off']:.6f}, on {KA3['step1_on']:.6f}). DoTA-CAP-dev is IN-DOMAIN for every row "
         f'(566 / 569 dev clips share a YouTube id with train).', '',
         '| row | train | input |', '|---|---|---|']
for row in ROWS:
    gpu = (row_paths(row)[0] / constants.KIM_RUN_ENV_FILENAME)
    gpu = json.loads(gpu.read_text())['gpu'] if gpu.exists() else 'not recorded'
    what = {ON: 'VideoMAE V2-S + KIM', K0: 'VideoMAE V2-S, KIM off', L0: 'LaGoVAD trunk (CLIP only)'}[row]
    lines.append(f"| {row} | {what} | {gpu} |")
lines += ['', '## Cells (macro [95 %] / micro / AP; micro and AP after per-clip min-max)', '',
          '| row | ' + ' | '.join(SET_NAME[s] for s in SETS) + ' |', '|---|---|---|']
for row in ROWS:
    cs = []
    for s in SETS:
        c = tab['motion'][s]['cells'][row]
        cs.append(f"{c['macro']['mean']:.4f} [{c['macro']['low']:.4f}, {c['macro']['high']:.4f}] / {c['micro']:.4f} / {c['ap']:.4f}")
    lines.append(f'| {row} | ' + ' | '.join(cs) + ' |')
lines += [f'| *position only (no pixels)* | *cubic ruler {pr:.4f} [{pl:.4f}, {ph:.4f}]* | *middle ruler {dr:.4f} [{dl:.4f}, {dh:.4f}]* |', '',
          '## Contrasts (paired; labels §53 KA5: GAIN ≥ +0.01 and interval above 0, COST mirror, else NO DIFFERENCE)', '',
          '| component | contrast | set | Δ micro | Δ macro |', '|---|---|---|---|---|']
for name, text, key, arm, unmatched in CONTRASTS:
    for s in SETS:
        d = tab[key][s]['deltas'][arm]
        lines.append(f"| {name} | {text} | {SET_NAME[s]} | {cell(d['micro'], unmatched)} | {cell(d['macro'], unmatched)} |")
lines += ['', 'Nexar-trained reference (printed, other corpus): §53.1 K5dfi KIM on − off micro DoTA −0.0041 n.s., DADA +0.0229; '
          '§36.1 K5dfi − L0dfi (motion + KIM) macro DoTA-CAP (whole) +0.147, DADA +0.193.', '']
for key, n in TABLES.items():
    lines += [(local / f'{constants.ZS_ARM_TABLE_STEM}_{n}.md').read_text(), '']
REPORT = local / f'{constants.KIM_DOTA_TRAIN_ABL_REPORT_STEM}.md'
REPORT.write_text('\n'.join(lines) + '\n')
(local / 'ka3.json').write_text(json.dumps(KA3, indent=1))
publish(local, REPORTS / 'ablation')
publish(REPORT, REPORTS / REPORT.name)
print((REPORTS / REPORT.name).read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/k5dfidota_ablation_report.md to Claude')